# Daily_station_metrics table

In this notebook I will populate the daily_metrics table. It has the following columns:
metric_id serial [pk]
- station_id varchar(50) [not null, ref: > stations.station_id]
- summary_date date [not null]
- trips_started integer [default: 0]
- trips_ended integer [default: 0]
- Net_flow:  trips_started - trips_ended
- Avg_bikes_available =  AVG(num_bikes_available) from availability_snapshot table by day
- min_bikes_available 
- max_bikes_available 
- Pct_time_empty = percent of times snapshots.num_bikes_available was 0 in a day
- pct_time_full  = percent of times


In [ ]:
import pandas as pd
import json
import os
import psycopg2
from psycopg2.extras import execute_values
import requests



## Connect to the citibike database on the Lightsail instance

In [ ]:
DB_HOST=os.environ.get('DB_HOST')
DB_NAME=os.environ.get('DB_NAME')
DB_USER=os.environ.get('DB_USER')
DB_PASSW=os.environ.get('DB_PASSWORD')
DB_PORT=os.environ.get('DB_PORT')

In [ ]:
#Connect to database citikike
try:
    conn = psycopg2.connect(
            host = DB_HOST,
            database = DB_NAME,
            user = DB_USER,
            password = DB_PASSW,
            port = DB_PORT,
            keepalives=1,
            keepalives_idle=30,
            keepalives_interval=10,
            keepalives_count=5
    )
    print('Connection Successful')

    cursor = conn.cursor()
except Exception as e:
    print(f'Could not establish connection:{e}')

These are the columns
                station_id, 
                summary_date,
                trips_started,
                trips_ended,
                Net_flow,  -- trips_started - trips_ended
                Avg_bikes_available, -- =  AVG(num_bikes_available) from availability_snapshot table by day
                min_bikes_available ,
                max_bikes_available, 
                Pct_time_empty, -- = percent of times snapshots.num_bikes_available was 0 in a day
                pct_time_full    --  = percent of times




### Calculate trips started at a station per day


In [ ]:
cursor.execute(
    '''
    SELECT 
        s.station_id, 
        DATE(t.started_at) AS summary_date, 
        COUNT(*) AS trips_started
    FROM trips t
    JOIN stations s on t.start_station_id = s.short_name
    GROUP BY s.station_id, DATE(t.started_at)
    LIMIT 10;
    ''')
cursor.fetchall()

### Calculate trips ending at a station per day

In [ ]:
cursor.execute(
    '''
    SELECT 
        s.station_id, 
        DATE(t.ended_at) AS summary_date, 
        COUNT(*) AS trips_ended
    FROM trips t
    JOIN stations s on t.end_station_id = s.short_name
    GROUP BY s.station_id, DATE(t.ended_at)
    LIMIT 10;
    ''')
cursor.fetchall()

### Availability metrics from availability_snapshots table

In [ ]:
# group by day and count min max bikes available
cursor.execute(
    '''
    SELECT 
        station_id, 
        DATE(captured_at) AS summary_date,
        AVG(num_bikes_available) AS avg_bikes_available,
        MIN(num_bikes_available) AS min_bikes_available, 
        MAX(num_bikes_available) AS max_bikes_available, 
        100* SUM(CASE WHEN num_bikes_available = 0 THEN 1 else 0 END) / COUNT(*)  AS pct_time_empty,
        100 * SUM(CASE WHEN num_docks_available = 0 THEN 1 else 0 END) / COUNT(*)  AS pct_time_full
    FROM availability_snapshots
    GROUP BY station_id, DATE(captured_at)
    limit 20;
   ''')
cursor.fetchall()    

- Create the CTEs

In [ ]:
### Put them together 
with trip_starts AS(
       SELECT 
        s.station_id, 
        DATE(t.started_at) AS summary_date, 
        COUNT(*) AS trips_started
    FROM trips t
    JOIN stations s on t.start_station_id = s.short_name
    GROUP BY s.station_id, DATE(t.started_at)
),
trip_ends AS(
        SELECT 
        s.station_id, 
        DATE(t.ended_at) AS summary_data, 
        COUNT(*) AS trips_ended
    FROM trips t
    JOIN stations s on t.end_station_id = s.short_name
    GROUP BY s.station_id, DATE(t.ended_at)
),
availability AS (
        SELECT 
        station_id, 
        DATE(captured_at) AS summary_date,
        AVG(num_bikes_available) AS avg_bikes_available,
        MIN(num_bikes_available) AS min_bikes_available, 
        MAX(num_bikes_available) AS max_bikes_available, 
        100* SUM(CASE WHEN num_bikes_available = 0 THEN 1 else 0 END) / COUNT(*)  AS pct_time_empty,
        100 * SUM(CASE WHEN num_docks_available = 0 THEN 1 else 0 END) / COUNT(*)  AS pct_time_full
    FROM availability_snapshots
    GROUP BY station_id, DATE(captured_at)
)


Since the live data starts in January, 
- there are December stations and HIST stationsthat will not have availability_snapshots(GBFS),
- also there are stations that are silent on certain days and do not have any trips but have snapshots,

because of this I need a join that preserves all the stations, not just the ones that exist in all the three CTE tables. Because of this I need a full outer join that preerves all the data

Also to join the start and end joins a station needs to exist in one of the start or end table for availability table to join on, so I need COALESE so that it returns the non null value

In [ ]:
COALESCE(s.station_id,e.station_id,av.station_id) ==> a station can exist in the GBFS data but have no trips
COALESCE(s.summary_date,e.summary_date, av.summary_date),
COALESCE(s.trips_started,0)
COALESCE(e.trips_ended,0)
COALESCE(s.trips_started,0) - COALESCE(e.trips_ended,0) AS net_flow 
av.avg_bikes_available,
av.min_bikes_available,
av.max_bikes_available,
av.pct_time_empty
av.pct_time_full



FROM trip_starts s
FULL OUTER JOIN trip_ends e ON s.station_id = e.station_id 
    AND s.summary_date = e.summary_date
FULL OUTER JOIN availability av ON COALESCE(S.station_id,e.station_id) = av.station_id
    AND COALESCE(e.summary_date,av.summary_date) = av.summary_date
GROUP BY station_id, summary_date

- Putting it all together

In [ ]:
INSERT INTO daily_station_metrics (
    station_id,
    summary_date,
    trips_started,
    trips_ended,
    avg_bikes_available,
    min_bikes_available,
    max_bikes_available,
    pct_time_empty,
    pct_time_full
)
with trip_starts AS(
       SELECT 
        s.station_id, 
        DATE(t.started_at) AS summary_date, 
        COUNT(*) AS trips_started
    FROM trips t
    JOIN stations s on t.start_station_id = s.short_name
    GROUP BY s.station_id, DATE(t.started_at)
),
trip_ends AS(
       SELECT 
        s.station_id, 
        DATE(t.ended_at) AS summary_date, 
        COUNT(*) AS trips_ended
    FROM trips t
    JOIN stations s on t.end_station_id = s.short_name
    GROUP BY s.station_id, DATE(t.ended_at)
),
availability AS (
        SELECT 
        station_id, 
        DATE(captured_at) AS summary_date,
        AVG(num_bikes_available) AS avg_bikes_available,
        MIN(num_bikes_available) AS min_bikes_available, 
        MAX(num_bikes_available) AS max_bikes_available, 
        100.0* SUM(CASE WHEN num_bikes_available = 0 THEN 1 else 0 END) / COUNT(*)  AS pct_time_empty,
        100.0 * SUM(CASE WHEN num_docks_available = 0 THEN 1 else 0 END) / COUNT(*)  AS pct_time_full
    FROM availability_snapshots
    GROUP BY station_id, DATE(captured_at)
)
SELECT
    COALESCE(s.station_id,e.station_id,av.station_id),  
    COALESCE(s.summary_date,e.summary_date, av.summary_date),
    COALESCE(s.trips_started,0),
    COALESCE(e.trips_ended,0),
    av.avg_bikes_available,
    av.min_bikes_available,
    av.max_bikes_available,
    av.pct_time_empty,
    Av.pct_time_full
FROM trip_starts s
FULL OUTER JOIN trip_ends e ON s.station_id = e.station_id 
    AND s.summary_date = e.summary_date
FULL OUTER JOIN availability av ON COALESCE(s.station_id,e.station_id) = av.station_id
    AND COALESCE(s.summary_date,e.summary_date) = av.summary_date;


In [ ]:
conn.close()
print('Connection is closed')